# MURA mirror vs official Stanford AIMI file index
Attach `cjinny/mura-v11`. Compares size and the first 48 bits of the MD5 of every PNG with the official Redivis `MURA-v1.1` index (embedded, compressed). Accelerator: None.

In [ ]:
import os, hashlib, gzip, base64, collections, time
OFFICIAL_B64 = "..."
official = {}
for ln in gzip.decompress(base64.b64decode(OFFICIAL_B64)).decode().split("\n"):
    p, s, m = ln.split("\t"); official[p] = (int(s), m)
print("official PNG entries:", len(official), "(md5 compared on the first 12 hex digits = 48 bits)")
root = None
for cur, dirs, files in os.walk("/kaggle/input"):
    if "train" in dirs and "valid" in dirs: root = cur; break
print("mirror root:", root)
t0 = time.time(); mirror = {}
for sp in ("train", "valid"):
    for cur, dirs, files in os.walk(os.path.join(root, sp)):
        for f in files:
            if not f.lower().endswith(".png") or f.startswith("._"): continue
            fp = os.path.join(cur, f); rel = os.path.relpath(fp, root).replace(os.sep, "/")
            h = hashlib.md5()
            with open(fp, "rb") as fh:
                for chunk in iter(lambda: fh.read(1 << 20), b""): h.update(chunk)
            mirror[rel] = (os.path.getsize(fp), h.hexdigest()[:12])
print("mirror PNG files:", len(mirror), "seconds:", round(time.time() - t0))
common = set(official) & set(mirror)
same = sum(1 for k in common if official[k] == mirror[k])
diff_size = [k for k in common if official[k][0] != mirror[k][0]]
diff_md5 = [k for k in common if official[k][0] == mirror[k][0] and official[k][1] != mirror[k][1]]
print("only in official:", len(set(official) - set(mirror)), "| only in mirror:", len(set(mirror) - set(official)))
print("in both:", len(common), "| identical size+md5:", same, "| size differs:", len(diff_size), "| md5 differs (same size):", len(diff_md5))
print("examples only-official:", sorted(set(official) - set(mirror))[:5]); print("examples only-mirror:", sorted(set(mirror) - set(official))[:5]); print("examples differing:", (diff_size + diff_md5)[:5])
by = collections.Counter(k.split("/")[0] for k in common if official[k] == mirror[k]); print("identical by split:", dict(by))

